## Configurando o Ambiente do Colab

###### OBS: Configurar o tipo da Runtime como GPU

In [1]:
"""Configuração Inicial do Ambiente"""

# Instalar dependências principais
!pip install ultralytics kaggle --quiet

import ultralytics
ultralytics.checks()


Ultralytics 8.3.235 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 38.1/112.6 GB disk)


In [2]:
"""Upload do kaggle.json"""

from google.colab import files

print("Envie o arquivo kaggle.json baixado do Kaggle:")
uploaded = files.upload()

kaggle_file = list(uploaded.keys())[0]

# Criar pasta e mover arquivo
!mkdir -p ~/.kaggle
!cp {kaggle_file} ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

print("kaggle.json configurado com sucesso!")
!ls -l ~/.kaggle



Envie o arquivo kaggle.json baixado do Kaggle:


Saving kaggle.json to kaggle.json
kaggle.json configurado com sucesso!
total 4
-rw------- 1 root root 72 Dec  8 13:19 kaggle.json


## Organizando o Dataset

In [3]:
"""Download e Extração do Dataset"""

# Baixar dataset do Kaggle
!kaggle datasets download -d gabrielferrante/brazilian-road-animals-bra-dataset

# Extrair para pasta dataset_raw/
!unzip -q brazilian-road-animals-bra-dataset.zip -d dataset_raw

# Conferir conteúdo
!ls dataset_raw


Dataset URL: https://www.kaggle.com/datasets/gabrielferrante/brazilian-road-animals-bra-dataset
License(s): CC0-1.0
 99% 451M/455M [00:01<00:00, 364MB/s]
100% 455M/455M [00:01<00:00, 459MB/s]
createTXT.py  Dataset	      divideValid.py   moveLabelsForLabelsFolder.py
dataAug.py    divideTrain.py  getXMLlabels.py  README.md


In [4]:
"""Preparação do Dataset para YOLOv8"""

import os
import shutil

# Criar estrutura padrão YOLOv8
for split in ["train", "val", "test"]:
    os.makedirs(f"bra-dataset/images/{split}", exist_ok=True)
    os.makedirs(f"bra-dataset/labels/{split}", exist_ok=True)

# Copiar imagens
shutil.copytree("dataset_raw/Dataset/images/train", "bra-dataset/images/train", dirs_exist_ok=True)
shutil.copytree("dataset_raw/Dataset/images/val", "bra-dataset/images/val", dirs_exist_ok=True)

# Copiar labels
shutil.copytree("dataset_raw/Dataset/labels/train", "bra-dataset/labels/train", dirs_exist_ok=True)
shutil.copytree("dataset_raw/Dataset/labels/val", "bra-dataset/labels/val", dirs_exist_ok=True)

print("Dataset preparado!")


Dataset preparado!


In [5]:
"""Verificar estrutura do Dataset"""

!apt-get install tree -y --quiet
!tree bra-dataset -L 3

# Verificar contagem
import os
print("Train Images:", len(os.listdir("bra-dataset/images/train")))
print("Val Images:", len(os.listdir("bra-dataset/images/val")))


Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  tree
0 upgraded, 1 newly installed, 0 to remove and 41 not upgraded.
Need to get 47.9 kB of archives.
After this operation, 116 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tree amd64 2.0.2-1 [47.9 kB]
Fetched 47.9 kB in 1s (87.2 kB/s)
Selecting previously unselected package tree.
(Reading database ... 121713 files and directories currently installed.)
Preparing to unpack .../tree_2.0.2-1_amd64.deb ...
Unpacking tree (2.0.2-1) ...
Setting up tree (2.0.2-1) ...
Processing triggers for man-db (2.10.2-1) ...
bra-dataset
├── images
│   ├── test
│   ├── train
│   │   ├── anta00000000.jpg
│   │   ├── anta00000001.jpg
│   │   ├── anta00000002.jpg
│   │   ├── anta00000003.jpg
│   │   ├── anta00000004.jpg
│   │   ├── anta00000005.jpg
│   │   ├── anta00000006.jpg
│   │   ├── anta00000007.jpg
│   │   ├── anta0000

In [6]:
"""Verificar as classes do dataset"""

with open("dataset_raw/Dataset/classes.txt") as f:
    classes = f.read().splitlines()

print("Classes detectadas:")
for i, c in enumerate(classes):
    print(i, c)


Classes detectadas:
0 Anta
1 Jaguarundi
2 LoboGuara
3 OncaParda
4 TamanduaBandeira


In [7]:
"""YAML de Configuração do Dataset"""

yaml_content = """path: /content/bra-dataset
train: images/train
val: images/val
test: images/test

names:
  0: Anta
  1: Jaguarundi
  2: LoboGuara
  3: OncaParda
  4: TamanduaBandeira
"""

for i, c in enumerate(classes):
    yaml_content += f"\n  {i}: {c}"

with open("bra-dataset.yaml", "w") as f:
    f.write(yaml_content)

print("bra-dataset.yaml criado:")
print(yaml_content)

bra-dataset.yaml criado:
path: /content/bra-dataset
train: images/train
val: images/val
test: images/test

names:
  0: Anta
  1: Jaguarundi
  2: LoboGuara
  3: OncaParda
  4: TamanduaBandeira

  0: Anta
  1: Jaguarundi
  2: LoboGuara
  3: OncaParda
  4: TamanduaBandeira


In [8]:
"""Verificar arquivo .yaml"""

with open("bra-dataset.yaml", "r") as f:
    print("Conteúdo do bra-dataset.yaml:")
    print(f.read())

Conteúdo do bra-dataset.yaml:
path: /content/bra-dataset
train: images/train
val: images/val
test: images/test

names:
  0: Anta
  1: Jaguarundi
  2: LoboGuara
  3: OncaParda
  4: TamanduaBandeira

  0: Anta
  1: Jaguarundi
  2: LoboGuara
  3: OncaParda
  4: TamanduaBandeira


## Iniciando o Treinamento

In [9]:
"""Treinamento Inicial"""
# (10 épocas para teste)
!yolo detect train data=bra-dataset.yaml model=yolov8n.pt epochs=10 imgsz=640


Ultralytics 8.3.235 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=bra-dataset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0, pretrai

In [10]:
"""Treinamento Final"""
# (50 épocas)

from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data="bra-dataset.yaml",
    epochs=50,
    imgsz=640,
    workers=2,
)



Ultralytics 8.3.235 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=bra-dataset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0, pretra

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x78176a644e30>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
        

In [11]:
"""Avaliação e Teste do Modelo"""

# Verificar arquivos gerados
!ls runs/detect/train/weights

# Verificar pesos gerados
!ls runs/detect/train/weights

# Testar com imagens de validação
model = YOLO("runs/detect/train/weights/best.pt")
model.predict(source="bra-dataset/images/val", save=True)

# Mostrar imagem anotada
from IPython.display import Image
Image(filename="runs/detect/predict/tamanduaBandeira00000377.jpg")

# Carregar melhor modelo
model = YOLO("runs/detect/train/weights/best.pt")

# Avaliar métricas
model.val()



best.pt  last.pt
best.pt  last.pt

image 1/363 /content/bra-dataset/images/val/anta00000345.jpg: 640x480 1 Anta, 37.4ms
image 2/363 /content/bra-dataset/images/val/anta00000368.jpg: 448x640 1 Anta, 38.6ms
image 3/363 /content/bra-dataset/images/val/anta00000391.jpg: 384x640 1 Anta, 37.1ms
image 4/363 /content/bra-dataset/images/val/anta00000396.jpg: 480x640 1 OncaParda, 37.4ms
image 5/363 /content/bra-dataset/images/val/anta00000397.jpg: 384x640 1 Anta, 6.4ms
image 6/363 /content/bra-dataset/images/val/anta00000398.jpg: 480x640 1 Anta, 6.3ms
image 7/363 /content/bra-dataset/images/val/anta00000399.jpg: 448x640 (no detections), 6.4ms
image 8/363 /content/bra-dataset/images/val/anta00000403.jpg: 416x640 3 Antas, 37.8ms
image 9/363 /content/bra-dataset/images/val/anta00000404.jpg: 448x640 1 Anta, 6.3ms
image 10/363 /content/bra-dataset/images/val/anta00000405.jpg: 384x640 1 Anta, 6.2ms
image 11/363 /content/bra-dataset/images/val/anta00000408.jpg: 384x640 1 Anta, 5.5ms
image 12/363 /conte

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x78176b249ee0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
        

## Salvando O Modelo Treinado

In [13]:
"""Montar Google Drive"""
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [14]:
"""Criar pasta no Drive para o projeto"""
!mkdir -p /content/drive/MyDrive/Projeto_IA_Fauna_BR/modelos


In [16]:
"""Copiar o best.pt para o Google Drive"""
!cp runs/detect/train/weights/best.pt /content/drive/MyDrive/Projeto_IA_Fauna_BR/modelos/best_fauna_br.pt

print("Modelo salvo com sucesso no Google Drive!")


Modelo salvo com sucesso no Google Drive!


In [17]:
"""Salvar também o last.pt"""
!cp runs/detect/train/weights/last.pt /content/drive/MyDrive/Projeto_IA_Fauna_BR/modelos/last_fauna_br.pt


In [18]:
"""Fazer download direto para o computador"""
from google.colab import files
files.download("runs/detect/train/weights/best.pt")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
"""Salvar todo o diretório de resultados do treinamento"""
!zip -r treino_completo.zip runs/detect/train
files.download("treino_completo.zip")


  adding: runs/detect/train/ (stored 0%)
  adding: runs/detect/train/val_batch1_labels.jpg (deflated 7%)
  adding: runs/detect/train/BoxR_curve.png (deflated 9%)
  adding: runs/detect/train/train_batch2.jpg (deflated 10%)
  adding: runs/detect/train/val_batch2_labels.jpg (deflated 7%)
  adding: runs/detect/train/val_batch1_pred.jpg (deflated 7%)
  adding: runs/detect/train/BoxPR_curve.png (deflated 11%)
  adding: runs/detect/train/confusion_matrix.png (deflated 22%)
  adding: runs/detect/train/confusion_matrix_normalized.png (deflated 19%)
  adding: runs/detect/train/train_batch1.jpg (deflated 9%)
  adding: runs/detect/train/labels.jpg (deflated 24%)
  adding: runs/detect/train/val_batch2_pred.jpg (deflated 6%)
  adding: runs/detect/train/val_batch0_labels.jpg (deflated 9%)
  adding: runs/detect/train/results.csv (deflated 57%)
  adding: runs/detect/train/BoxF1_curve.png (deflated 8%)
  adding: runs/detect/train/weights/ (stored 0%)
  adding: runs/detect/train/weights/best.pt (deflated

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Testes com Imagens e Vídeos

In [20]:
"""Testar com Imagem de Validação"""
model.predict(source="bra-dataset/images/val", save=True)



image 1/363 /content/bra-dataset/images/val/anta00000345.jpg: 640x480 1 Anta, 9.5ms
image 2/363 /content/bra-dataset/images/val/anta00000368.jpg: 448x640 1 Anta, 8.5ms
image 3/363 /content/bra-dataset/images/val/anta00000391.jpg: 384x640 1 Anta, 9.8ms
image 4/363 /content/bra-dataset/images/val/anta00000396.jpg: 480x640 1 OncaParda, 8.4ms
image 5/363 /content/bra-dataset/images/val/anta00000397.jpg: 384x640 1 Anta, 11.0ms
image 6/363 /content/bra-dataset/images/val/anta00000398.jpg: 480x640 1 Anta, 9.4ms
image 7/363 /content/bra-dataset/images/val/anta00000399.jpg: 448x640 (no detections), 8.5ms
image 8/363 /content/bra-dataset/images/val/anta00000403.jpg: 416x640 3 Antas, 8.6ms
image 9/363 /content/bra-dataset/images/val/anta00000404.jpg: 448x640 1 Anta, 11.0ms
image 10/363 /content/bra-dataset/images/val/anta00000405.jpg: 384x640 1 Anta, 8.5ms
image 11/363 /content/bra-dataset/images/val/anta00000408.jpg: 384x640 1 Anta, 8.4ms
image 12/363 /content/bra-dataset/images/val/anta0000040

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'Anta', 1: 'Jaguarundi', 2: 'LoboGuara', 3: 'OncaParda', 4: 'TamanduaBandeira'}
 obb: None
 orig_img: array([[[ 80,  95,  98],
         [ 87, 100, 102],
         [ 91, 103, 105],
         ...,
         [ 70,  78,  91],
         [ 70,  78,  91],
         [ 71,  77,  90]],
 
        [[ 80,  94, 100],
         [ 82,  94,  98],
         [ 83,  94,  98],
         ...,
         [ 68,  82,  94],
         [ 68,  80,  92],
         [ 69,  81,  93]],
 
        [[ 77,  88,  96],
         [ 75,  87,  93],
         [ 75,  85,  92],
         ...,
         [ 70,  82,  94],
         [ 66,  78,  90],
         [ 69,  81,  93]],
 
        ...,
 
        [[ 25,  27,  27],
         [ 38,  40,  40],
         [ 46,  48,  48],
         ...,
         [ 82,  80,  79],
         [ 80,  81,  79],
         [ 69,  71,  72]],
 
        [[ 35,  38,  42],
         [ 46,

In [27]:
"""Testar o Modelo com Imagem Local"""
from google.colab import files
uploaded = files.upload()

file = list(uploaded.keys())[0]

model.predict(source=file, save=True)



Saving images.webp to images.webp

image 1/1 /content/images.webp: 640x640 1 LoboGuara, 10.8ms
Speed: 7.7ms preprocess, 10.8ms inference, 2.2ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /content/runs/detect/predict2


[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'Anta', 1: 'Jaguarundi', 2: 'LoboGuara', 3: 'OncaParda', 4: 'TamanduaBandeira'}
 obb: None
 orig_img: array([[[135, 135, 128],
         [130, 130, 123],
         [123, 123, 116],
         ...,
         [118, 122, 110],
         [120, 124, 112],
         [121, 125, 113]],
 
        [[135, 135, 128],
         [130, 130, 123],
         [123, 123, 116],
         ...,
         [116, 121, 108],
         [119, 123, 111],
         [120, 124, 112]],
 
        [[135, 135, 128],
         [130, 130, 123],
         [123, 123, 116],
         ...,
         [114, 118, 106],
         [116, 121, 108],
         [118, 122, 110]],
 
        ...,
 
        [[ 92, 143, 128],
         [ 91, 142, 127],
         [ 99, 145, 130],
         ...,
         [125, 107, 112],
         [125, 107, 112],
         [124, 106, 111]],
 
        [[ 94, 142, 130],
         [ 93,

In [22]:
"""Testar com Imagem da Internet"""

!wget https://ultralytics.com/images/bus.jpg -O test.jpg
model.predict(source="test.jpg", save=True)


--2025-12-08 14:07:49--  https://ultralytics.com/images/bus.jpg
Resolving ultralytics.com (ultralytics.com)... 198.202.211.1
Connecting to ultralytics.com (ultralytics.com)|198.202.211.1|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://www.ultralytics.com/images/bus.jpg [following]
--2025-12-08 14:07:49--  https://www.ultralytics.com/images/bus.jpg
Resolving www.ultralytics.com (www.ultralytics.com)... 198.202.211.1, 2620:cb:2000::1
Connecting to www.ultralytics.com (www.ultralytics.com)|198.202.211.1|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://github.com/ultralytics/assets/releases/download/v0.0.0/bus.jpg [following]
--2025-12-08 14:07:50--  https://github.com/ultralytics/assets/releases/download/v0.0.0/bus.jpg
Resolving github.com (github.com)... 140.82.116.4
Connecting to github.com (github.com)|140.82.116.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Loc

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'Anta', 1: 'Jaguarundi', 2: 'LoboGuara', 3: 'OncaParda', 4: 'TamanduaBandeira'}
 obb: None
 orig_img: array([[[119, 146, 172],
         [121, 148, 174],
         [122, 152, 177],
         ...,
         [161, 171, 188],
         [160, 170, 187],
         [160, 170, 187]],
 
        [[120, 147, 173],
         [122, 149, 175],
         [123, 153, 178],
         ...,
         [161, 171, 188],
         [160, 170, 187],
         [160, 170, 187]],
 
        [[123, 150, 176],
         [124, 151, 177],
         [125, 155, 180],
         ...,
         [161, 171, 188],
         [160, 170, 187],
         [160, 170, 187]],
 
        ...,
 
        [[183, 182, 186],
         [179, 178, 182],
         [180, 179, 183],
         ...,
         [121, 111, 117],
         [113, 103, 109],
         [115, 105, 111]],
 
        [[165, 164, 168],
         [173,

In [23]:
"""Testar com Vídeo Local"""
uploaded = files.upload()
video_file = list(uploaded.keys())[0]

model.predict(source=video_file, save=True, conf=0.25)



IndexError: list index out of range

In [24]:
"""Testar com Vídeo da Internet"""

!wget https://ultralytics.com/images/traffic.mp4 -O traffic.mp4
model.predict(source="traffic.mp4", save=True, conf=0.25)



--2025-12-08 14:09:06--  https://ultralytics.com/images/traffic.mp4
Resolving ultralytics.com (ultralytics.com)... 198.202.211.1
Connecting to ultralytics.com (ultralytics.com)|198.202.211.1|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://www.ultralytics.com/images/traffic.mp4 [following]
--2025-12-08 14:09:06--  https://www.ultralytics.com/images/traffic.mp4
Resolving www.ultralytics.com (www.ultralytics.com)... 198.202.211.1, 2620:cb:2000::1
Connecting to www.ultralytics.com (www.ultralytics.com)|198.202.211.1|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://github.com/ultralytics/assets/releases/download/v0.0.0/traffic.mp4 [following]
--2025-12-08 14:09:06--  https://github.com/ultralytics/assets/releases/download/v0.0.0/traffic.mp4
Resolving github.com (github.com)... 140.82.116.4
Connecting to github.com (github.com)|140.82.116.4|:443... connected.
HTTP request sent, awaiting respo

FileNotFoundError: Failed to open video /content/traffic.mp4

In [25]:
"""Visualizar um Frame Detectado"""

from IPython.display import HTML
from base64 import b64encode

predict_dirs = sorted([d for d in os.listdir("runs/detect") if "predict" in d])
latest = f"runs/detect/{predict_dirs[-1]}"

video_path = f"{latest}/video.mp4"

mp4 = open(video_path,'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

HTML(f"""
<video width=640 controls>
    <source src="{data_url}" type="video/mp4">
</video>
""")


FileNotFoundError: [Errno 2] No such file or directory: 'runs/detect/predict2/video.mp4'

In [26]:
"""Visualizar Vídeo no Colab"""
from IPython.display import HTML
from base64 import b64encode

video_path = f"{latest}/video.mp4"

mp4 = open(video_path,'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

HTML(f"""
<video width=640 controls>
    <source src="{data_url}" type="video/mp4">
</video>
""")


FileNotFoundError: [Errno 2] No such file or directory: 'runs/detect/predict2/video.mp4'